# UAE Mobile Network Experience Intelligence — Ookla Dataset Overview

**Purpose of this notebook:** a walkthrough for the team of what we're building, why the Ookla
dataset behaves the way it does, and the gotchas that already bit us once (and will bite anyone who
skips this). Read this before writing any code that touches Ookla data.

For the actual production extraction pipeline (all 8 quarters, saved to `data/processed/`), see
[`01_ookla_collection.ipynb`](01_ookla_collection.ipynb). This notebook re-derives a few of the same
numbers on purpose, so the claims below are backed by code you can re-run, not just prose.

## 1. What we're building

**The challenge:** use public, outside-in mobile measurements to build an AI-powered UAE Mobile
Network Experience Intelligence Map that identifies areas of potential concern, detects meaningful
change, prioritizes where investigation would create the most value, and lets decision-makers
interrogate the evidence in plain language.

**What this is not:** a tool for diagnosing e&'s network. The public dataset carries no operator
attribution, so we can never claim to know *why* a zone is weak from an operator's point of view —
only that it appears weak, relative to comparable peers, with a stated confidence. If asked "is this
e& data?" or "which e& site is causing this?", the correct answer is always a grounded refusal — the
data can't support that claim. This boundary is deliberate and matters to the judging panel.

**The journey:** `Data → Map → Detection → Prioritization → Explanation → Decision`

**Three mandatory public datasets**, all clipped to the real UAE boundary (never a lon/lat box —
see §4 for why):

| Dataset | What it gives us | Status |
|---|---|---|
| Ookla Speedtest Open Data (mobile) | Download/upload/latency per ~610m tile, quarterly | Done — this notebook + [`01_ookla_collection.ipynb`](01_ookla_collection.ipynb) |
| WorldPop UAE population | Estimated population per 100m cell, 2026 | Done — [`02_worldpop_collection.ipynb`](02_worldpop_collection.ipynb) |
| OpenStreetMap | Building/road/POI density for peer grouping | Raw extract downloaded, UAE feature extraction pending — [`03_osm_collection.ipynb`](03_osm_collection.ipynb) |

**Seven capabilities on top of that data** (mandatory, roughly in build order): interactive map,
Experience Index (0–100, deterministic formula we design), Confidence Score (0–100, from sample
size), ML anomaly detection (Peer Gap + Temporal Anomaly, benchmarked against a simple baseline),
change/trend intelligence (relative to peer trend, not raw Mbps), a geographic prioritization
engine, and a grounded AI copilot that narrates — but never calculates — the results.

**One rule that shapes everything downstream:** every number a user sees must be a traceable,
reproducible computation. The LLM only narrates what the deterministic formulas and ML models
already computed — it never invents or calculates a score itself.

In [1]:
import urllib.request

import duckdb
import pandas as pd
import geopandas as gpd
from pathlib import Path

## 2. The Ookla dataset, field by field

Each row is one **tile**: a fixed ~610m x 610m grid cell (at the equator; slightly smaller at UAE
latitude), aggregated over one calendar quarter from crowd-sourced Speedtest results.

| Field | Meaning | Watch out for |
|---|---|---|
| `quadkey` | Bing-style tile identifier, encodes tile position + implicit zoom level | Stable tile ID *within* a quarter — not guaranteed to recur across quarters (§7) |
| `tile_x`, `tile_y` | Tile centroid | **`tile_x` is longitude, `tile_y` is latitude.** Swap them and every point silently lands in the ocean or in Iran. |
| `avg_d_kbps`, `avg_u_kbps` | Mean download / upload speed | Units are **kbps**, not Mbps — divide by 1000 |
| `avg_lat_ms` | Mean unloaded (idle) latency, ms | The brief calls for *loaded* latency where available — see next row |
| `avg_lat_down_ms`, `avg_lat_up_ms` | Mean loaded latency during download / upload, ms | ~99% populated for UAE rows; not fully populated globally |
| `tests` | Number of individual speed tests in the tile that quarter | Median is very low per tile (§6) — this is the whole reason we need a Confidence Score |
| `devices` | Number of distinct devices contributing tests | Can be smaller than `tests` (repeat tests from the same device/user) |

Licence: **CC BY-NC-SA 4.0** (non-commercial). Source: Ookla Open Data, `teamookla/ookla-open-data`
on GitHub / AWS Registry of Open Data, `MOBILE` layer only (fixed broadband is out of scope for this
challenge).

In [2]:
# Load one quarter (2026 Q2, the latest published) straight from the raw parquet.
# This is the same file 01_ookla_collection.ipynb uses for the 8-quarter pipeline.
# Both the tile data and the UAE boundary are downloaded here if not already
# present, so this notebook runs standalone on a fresh clone.

RAW_DIR = Path("../data/raw/ookla")
RAW_DIR.mkdir(parents=True, exist_ok=True)

ookla_file = RAW_DIR / "2026-04-01_performance_mobile_tiles.parquet"
if not ookla_file.exists():
    url = (
        "https://ookla-open-data.s3.amazonaws.com/parquet/performance/"
        "type=mobile/year=2026/quarter=2/2026-04-01_performance_mobile_tiles.parquet"
    )
    print("Downloading:", url)
    urllib.request.urlretrieve(url, ookla_file)

BOUNDARY_URL = (
    "https://github.com/wmgeolab/geoBoundaries/raw/9469f09/"
    "releaseData/gbOpen/ARE/ADM0/geoBoundaries-ARE-ADM0.geojson"
)
boundary_file = Path("../data/raw/boundary/uae_boundary.geojson")
boundary_file.parent.mkdir(parents=True, exist_ok=True)
if not boundary_file.exists():
    print("Downloading UAE boundary (geoBoundaries.org, ADM0):", BOUNDARY_URL)
    urllib.request.urlretrieve(BOUNDARY_URL, boundary_file)

raw = duckdb.sql(f"SELECT * FROM read_parquet('{ookla_file}')").df()
print("Rows in the raw global file (before any UAE filtering):", len(raw))
raw[["quadkey", "tile_x", "tile_y", "avg_d_kbps", "avg_u_kbps", "avg_lat_ms", "tests", "devices"]].head()

Rows in the raw global file (before any UAE filtering): 3382601


,quadkey,tile_x,tile_y,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices
0,0022133222330201,-160.0406,70.6336,31623,17555,81,1,1
1,0022332203013331,-162.6004,66.8988,5931,9901,79,1,1
2,0022332203031111,-162.6004,66.8945,29027,17188,277,1,1
3,0022332203031131,-162.6004,66.8902,14526,3041,79,1,1
4,0022332203102220,-162.5949,66.8988,72487,25752,54,1,1


## 3. Gotcha #1 — never filter the UAE with a lon/lat bounding box

A bounding box around the UAE also catches Doha, parts of Oman, and open Gulf water. Below we
compare a rough bounding box against an exact point-in-polygon clip against the real UAE boundary,
on the same quarter.

In [3]:
uae_boundary = gpd.read_file(boundary_file).to_crs("EPSG:4326")

# Step 1: rough bounding-box pre-filter (cheap, but NOT the final answer)
bbox_candidates = duckdb.sql(f"""
    SELECT * FROM read_parquet('{ookla_file}')
    WHERE tile_x BETWEEN 51 AND 57
      AND tile_y BETWEEN 22 AND 27
""").df()

# Step 2: exact polygon clip
points = gpd.GeoDataFrame(
    bbox_candidates,
    geometry=gpd.points_from_xy(bbox_candidates["tile_x"], bbox_candidates["tile_y"]),
    crs="EPSG:4326",
)
uae_exact = gpd.sjoin(points, uae_boundary[["geometry"]], predicate="within", how="inner")

inflation = len(bbox_candidates) / len(uae_exact) - 1
print(f"Bounding-box candidates: {len(bbox_candidates):,}")
print(f"Exact UAE-boundary tiles: {len(uae_exact):,}")
print(f"Bounding box overcounts by {inflation:.0%} — that's Doha, Oman and Gulf water sneaking in.")

Bounding-box candidates: 10,367
Exact UAE-boundary tiles: 6,879
Bounding box overcounts by 51% — that's Doha, Oman and Gulf water sneaking in.


The brief cites roughly 30% inflation from a bounding box; we're seeing 51% on the 2026 Q2 file (coverage has grown since the brief was written, so the contaminating region scales too). Same direction, bigger gap — the point stands either way. **Always clip to `data/raw/boundary/uae_boundary.geojson`, never to a bounding box.**

## 4. Gotcha #2 — `tile_x` is longitude, `tile_y` is latitude

H3 (and most geospatial libraries) expect `(lat, lon)` order. Ookla's column names are easy to
misread the other way. We already got this right above (`points_from_xy(tile_x, tile_y)` — x is
always longitude), but it's worth saying explicitly once: **flip these and your zones land silently
in the sea, with no error to warn you.**

In [4]:
# Sanity check: UAE longitude range is ~51-57, latitude range is ~22-27.
# If this ever prints values outside those ranges, tile_x/tile_y have been swapped somewhere upstream.
print("tile_x (longitude) range:", uae_exact["tile_x"].min(), "-", uae_exact["tile_x"].max())
print("tile_y (latitude) range:", uae_exact["tile_y"].min(), "-", uae_exact["tile_y"].max())

tile_x (longitude) range: 51.5945 - 56.373599999999996
tile_y (latitude) range: 22.6723 - 26.0543


## 5. Units and derived metrics

Speeds are published in kbps. We convert to Mbps for anything human-facing (the Experience Index,
the map, the copilot). Latency is already in ms.

In [5]:
uae_exact["download_mbps"] = uae_exact["avg_d_kbps"] / 1000
uae_exact["upload_mbps"] = uae_exact["avg_u_kbps"] / 1000

uae_exact[["download_mbps", "upload_mbps", "avg_lat_ms", "avg_lat_down_ms", "avg_lat_up_ms", "tests", "devices"]].describe()

,download_mbps,upload_mbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices
count,6879.000000,6879.000000,6879.000000,6819.000000,6843.000000,6879.000000,6879.000000
mean,414.551638,31.237349,30.390173,610.744831,973.617273,5.992586,3.337404
std,361.460965,27.103569,38.112555,716.147802,970.591322,14.504537,8.164917
min,0.050000,0.002000,0.000000,21.000000,19.000000,1.000000,1.000000
25%,117.143500,11.477500,16.000000,300.000000,332.000000,1.000000,1.000000
50%,335.930000,25.053000,21.000000,404.000000,648.000000,2.000000,2.000000
75%,612.374500,42.702500,29.000000,617.500000,1257.500000,5.000000,3.000000
max,2193.000000,325.974000,684.000000,9907.000000,9619.000000,458.000000,324.000000


## 6. Gotcha #3 — individual tiles are sparse; that's *why* we aggregate and score confidence

At raw tile granularity (~610m), most tiles have almost no tests. This is not a data quality bug —
it's the nature of crowd-sourced measurement at fine spatial resolution, and it's exactly why the
brief requires a Confidence Score and aggregation to a coarser geographic unit (H3 resolution 6/7)
before scoring anything.

In [6]:
median_tests = uae_exact["tests"].median()
pct_below_30 = (uae_exact["tests"] < 30).mean()

print(f"Median tests per tile (2026 Q2): {median_tests:.0f}")
print(f"Share of tiles with fewer than 30 tests: {pct_below_30:.1%}")
print()
print("These numbers look sparser than the brief's zone-level figures (median 4 tests, 83% below 30)")
print("because the brief is describing aggregated ZONES (H3 res 6/7), not raw ~610m tiles.")
print("Individual tiles are even sparser — pooling them into zones is what makes the sample sizes workable.")

Median tests per tile (2026 Q2): 2
Share of tiles with fewer than 30 tests: 96.8%

These numbers look sparser than the brief's zone-level figures (median 4 tests, 83% below 30)
because the brief is describing aggregated ZONES (H3 res 6/7), not raw ~610m tiles.
Individual tiles are even sparser — pooling them into zones is what makes the sample sizes workable.


**Implication for the team:** don't compute the Experience Index or run anomaly detection on
raw tiles. Aggregate to the chosen H3 resolution first, and carry `tests`/`devices` through the
aggregation so the Confidence Score has something to work with. A zone with 3 tests from 2 devices
must never be displayed the same way as a zone with 500 tests from 300 devices — the honest answer
for the former is "insufficient public evidence", not "bad network".

## 7. Gotcha #4 — tiles churn across quarters; don't inner-join on quadkey across all 8

`quadkey` identifies a tile *within* a quarter's snapshot. It is not a stable ID you can join across
quarters — coverage shifts quarter to quarter as different people run tests in different places. We
verify this directly below by loading all 8 quarters' UAE-clipped quadkeys and checking how many
persist in every single one.

In [7]:
QUARTERS = [
    (2024, 3, "2024-07-01"),
    (2024, 4, "2024-10-01"),
    (2025, 1, "2025-01-01"),
    (2025, 2, "2025-04-01"),
    (2025, 3, "2025-07-01"),
    (2025, 4, "2025-10-01"),
    (2026, 1, "2026-01-01"),
    (2026, 2, "2026-04-01"),
]

ookla_files = []
for year, quarter, date_str in QUARTERS:
    local_path = RAW_DIR / f"{date_str}_performance_mobile_tiles.parquet"
    if not local_path.exists():
        url = (
            "https://ookla-open-data.s3.amazonaws.com/parquet/performance/"
            f"type=mobile/year={year}/quarter={quarter}/"
            f"{date_str}_performance_mobile_tiles.parquet"
        )
        print("Downloading:", url)
        urllib.request.urlretrieve(url, local_path)
    ookla_files.append(local_path)

def uae_quadkeys(file):
    cand = duckdb.sql(f"""
        SELECT quadkey, tile_x, tile_y FROM read_parquet('{file}')
        WHERE tile_x BETWEEN 51 AND 57 AND tile_y BETWEEN 22 AND 27
    """).df()
    pts = gpd.GeoDataFrame(cand, geometry=gpd.points_from_xy(cand["tile_x"], cand["tile_y"]), crs="EPSG:4326")
    joined = gpd.sjoin(pts, uae_boundary[["geometry"]], predicate="within", how="inner")
    return set(joined["quadkey"])

quarter_quadkey_sets = [uae_quadkeys(f) for f in ookla_files]

all_quadkeys = set().union(*quarter_quadkey_sets)
persistent_quadkeys = set.intersection(*quarter_quadkey_sets)

print(f"Unique UAE tiles seen across all 8 quarters: {len(all_quadkeys):,}")
print(f"Tiles present in EVERY quarter:              {len(persistent_quadkeys):,}")
print(f"Persistence rate: {len(persistent_quadkeys) / len(all_quadkeys):.1%}")

Unique UAE tiles seen across all 8 quarters: 15,201
Tiles present in EVERY quarter:              1,730
Persistence rate: 11.4%


This matches the brief's own finding almost exactly (~11%). An inner join on `quadkey` across
all 8 quarters would throw away close to 90% of the data. **The fix: aggregate tiles into zones
(H3) first, then compare zones across quarters — not raw tiles.**

## 8. Update — the quarter-labelling gap described above is now fixed

`01_ookla_collection.ipynb` now tags every row with a `quarter` column (e.g. `"2026Q2"`) before the
`pd.concat`, so `(quadkey, quarter)` is unique even though `quadkey` alone still repeats across
quarters. The notebook also now downloads the 8 quarters itself from the public `ookla-open-data`
S3 bucket rather than assuming the raw files already exist locally, and parses the raw `tile` WKT
column into a real `tile_geometry` polygon column instead of discarding it as a string.

One thing to know when reading the processed file yourself: it now has **two** GeoParquet geometry
columns (`geometry` = tile centroid point, `tile_geometry` = the real tile polygon boundary).
Read it with `geopandas.read_parquet(...)`, not `pandas.read_parquet(...)` — plain pandas returns
raw WKB bytes for both columns instead of usable geometries, silently.


In [8]:
print("Confirming the fix, reading the processed file correctly this time:")
processed = gpd.read_parquet("../data/processed/ookla_tiles_uae.parquet")
print("Total rows:", len(processed))
print("Unique quadkeys:", processed["quadkey"].nunique())
print("Rows sharing a quadkey with another row (expected -- persistent tiles across quarters):",
      processed.duplicated(subset=["quadkey"]).sum())
print("Rows sharing (quadkey, quarter) with another row (should be 0):",
      processed.duplicated(subset=["quadkey", "quarter"]).sum())

Confirming the fix, reading the processed file correctly this time:
Total rows: 50860
Unique quadkeys: 15201
Rows sharing a quadkey with another row (expected -- persistent tiles across quarters): 35659
Rows sharing (quadkey, quarter) with another row (should be 0): 0


## 9. Where this fits, and what's next

This notebook only covers Ookla. The full picture:

- **Ookla** (this notebook + [`01_ookla_collection.ipynb`](01_ookla_collection.ipynb)) — done, 8
  quarters, UAE-clipped, self-downloading, quarter-labelled.
- **WorldPop** ([`02_worldpop_collection.ipynb`](02_worldpop_collection.ipynb)) — done. National
  total (11,476,873) verified against the brief's ~11.5M.
- **OSM** ([`03_osm_collection.ipynb`](03_osm_collection.ipynb)) — raw GCC States PBF downloaded;
  pyosmium extraction of UAE buildings/roads/POIs/land-use (for peer grouping) still pending.

**Next up, roughly in order:** pick and justify the H3 resolution for zone aggregation (brief
suggests res 7, reserving res 8 for display only); build the Experience Index and Confidence Score;
then anomaly detection, trend intelligence, the priority engine, and finally the grounded copilot.